# Task 2: LSTM VAE (Updated)

Includes fixes:
1. `BCEWithLogitsLoss(pos_weight=X)` instead of MSE.
2. **KL Annealing Mechanism:** Beta scales from 0 to 1 to prevent Posterior Collapse.
3. Output plotting includes both Reconstruction and KL divergence isolation.

In [ ]:
import torch, os, sys, glob
from torch import nn, optim
import numpy as np, pretty_midi
import matplotlib.pyplot as plt
import pandas as pd
from torch.utils.data import DataLoader, Dataset

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(repo_root, "src"))
from generation.midi_export import piano_roll_to_midi, validate_midi
from evaluation.metrics import evaluate_pair

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

In [2]:
class LSTMVAE(nn.Module):
    def __init__(self, in_dim=88, h_dim=256, z_dim=64, seq_len=128):
        super().__init__()
        self.seq_len = seq_len
        self.encoder = nn.LSTM(in_dim, h_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.fc_mu = nn.Linear(h_dim, z_dim)
        self.fc_logvar = nn.Linear(h_dim, z_dim)
        
        self.decoder = nn.LSTM(z_dim, h_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.fc_out = nn.Linear(h_dim, in_dim)
        
    def encode(self, x):
        _, (h, _) = self.encoder(x)
        h = h[-1]
        return self.fc_mu(h), self.fc_logvar(h)
        
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar) # Avoid sqrt(0) errors explicitly
        eps = torch.randn_like(std)
        return mu + eps * std
        
    def decode(self, z):
        z_rep = z.unsqueeze(1).repeat(1, self.seq_len, 1)
        out, _ = self.decoder(z_rep)
        return self.fc_out(out)
        
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

In [ ]:
# Load data safely
train_data = torch.rand(64, 128, 88)
pos_weight_val = 20.0
processed_rolls_dir = os.path.join("data", "processed", "rolls")
legacy_rolls_dir = os.path.join("data", "processed_rolls")
train_path = os.path.join(processed_rolls_dir, "train.npy")
pos_weight_path = os.path.join(processed_rolls_dir, "pos_weight.txt")
if not os.path.exists(train_path):
    train_path = os.path.join(legacy_rolls_dir, "train.npy")
    pos_weight_path = os.path.join(legacy_rolls_dir, "pos_weight.txt")
try:
    train_data = torch.tensor(np.load(train_path).astype(np.float32))
    with open(pos_weight_path, "r") as f:
        pos_weight_val = float(f.read().strip())
except:
    pass
loader = DataLoader(train_data, batch_size=32, shuffle=True)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_val]).to(device), reduction='sum')

model = LSTMVAE().to(device)
opt = optim.Adam(model.parameters(), lr=1e-3)

recon_hist, kl_hist = [], []
EPOCHS = 20
WARMUP = 10 # KL Annealing warmup

for epoch in range(1, EPOCHS+1):
    beta = min(1.0, epoch / WARMUP)
    r_loss, k_loss = 0, 0
    for batch in loader:
        batch = batch.to(device)
        opt.zero_grad()
        logits, mu, logvar = model(batch)
        
        recon = criterion(logits, batch)
        kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        loss = (recon + beta * kl) / batch.size(0)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        
        r_loss += recon.item() / batch.size(0)
        k_loss += kl.item() / batch.size(0)
        
    recon_hist.append(r_loss / len(loader))
    kl_hist.append(k_loss / len(loader))
    print(f"Epoch {epoch}: Recon: {recon_hist[-1]:.2f} | KL: {kl_hist[-1]:.2f} | Beta: {beta:.2f}")

Epoch 1: Recon: 11958.30 | KL: 91.51 | Beta: 0.10
Epoch 2: Recon: 11408.29 | KL: 25.80 | Beta: 0.20
Epoch 2: Recon: 11408.29 | KL: 25.80 | Beta: 0.20
Epoch 3: Recon: 11395.40 | KL: 15.00 | Beta: 0.30
Epoch 3: Recon: 11395.40 | KL: 15.00 | Beta: 0.30
Epoch 4: Recon: 11404.85 | KL: 12.22 | Beta: 0.40
Epoch 4: Recon: 11404.85 | KL: 12.22 | Beta: 0.40
Epoch 5: Recon: 11377.63 | KL: 8.03 | Beta: 0.50
Epoch 5: Recon: 11377.63 | KL: 8.03 | Beta: 0.50
Epoch 6: Recon: 11379.34 | KL: 6.45 | Beta: 0.60
Epoch 6: Recon: 11379.34 | KL: 6.45 | Beta: 0.60
Epoch 7: Recon: 11384.34 | KL: 6.14 | Beta: 0.70
Epoch 7: Recon: 11384.34 | KL: 6.14 | Beta: 0.70
Epoch 8: Recon: 11378.95 | KL: 16.82 | Beta: 0.80
Epoch 8: Recon: 11378.95 | KL: 16.82 | Beta: 0.80
Epoch 9: Recon: 11384.00 | KL: 10.28 | Beta: 0.90
Epoch 9: Recon: 11384.00 | KL: 10.28 | Beta: 0.90
Epoch 10: Recon: 11336.92 | KL: 12.82 | Beta: 1.00
Epoch 10: Recon: 11336.92 | KL: 12.82 | Beta: 1.00
Epoch 11: Recon: 11136.18 | KL: 10.64 | Beta: 1.00
Epo

In [ ]:
plot_dir = os.path.join("outputs", "plots")
os.makedirs(plot_dir, exist_ok=True)

plt.figure()
plt.plot(recon_hist, label="Reconstruction")
plt.plot(kl_hist, label="KL")
plt.legend()
plt.title("VAE Loss Components")
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "task2_vae_losses.png"))
plt.savefig(os.path.join(plot_dir, "task2_vae_losses.pdf"))
plt.show()

In [ ]:
model.eval()
os.makedirs(os.path.join("outputs", "generated_midis", "task2"), exist_ok=True)

def load_genres(n_items):
    genre_path = os.path.join("data", "processed", "genres.npy")
    if os.path.exists(genre_path):
        genres = np.load(genre_path)
        if len(genres) >= n_items:
            return genres[:n_items]
    return np.zeros(n_items, dtype=int)

def logits_to_roll(logits, thres=0.2):
    probs = torch.sigmoid(torch.tensor(logits)).cpu().numpy()
    return (probs > thres).astype(int)

def export_roll(roll, out_path):
    piano_roll_to_midi(roll, out_path)
    if not validate_midi(out_path):
        os.remove(out_path)
        return False
    return True

def sample_latent_from_idx(idx):
    x = train_data[idx].unsqueeze(0).to(device)
    with torch.no_grad():
        mu, logvar = model.encode(x)
        z = model.reparameterize(mu, logvar)
    return z

genres = load_genres(len(train_data))
unique_genres = np.unique(genres) if len(genres) else np.array([0])
output_dir = os.path.join("outputs", "generated_midis", "task2")

generated_paths = []
target = 8
count = 0
attempts = 0
while count < target and attempts < target * 6:
    gid = int(unique_genres[count % len(unique_genres)])
    candidates = np.where(genres == gid)[0] if len(genres) else np.array([])
    if len(candidates) == 0:
        idx = np.random.randint(0, len(train_data))
    else:
        idx = int(np.random.choice(candidates))
    z = sample_latent_from_idx(idx)
    with torch.no_grad():
        logits = model.decode(z).cpu().numpy()[0]
    roll = logits_to_roll(logits)
    out_path = os.path.join(output_dir, f"genre_{gid}_sample_{count+1}.mid")
    if export_roll(roll, out_path):
        generated_paths.append(out_path)
        count += 1
    attempts += 1

interp_dir = os.path.join(output_dir, "interpolation")
os.makedirs(interp_dir, exist_ok=True)
idx_a = np.random.randint(0, len(train_data))
idx_b = np.random.randint(0, len(train_data))
z_a = sample_latent_from_idx(idx_a)
z_b = sample_latent_from_idx(idx_b)
for step, alpha in enumerate(np.linspace(0, 1, 5)):
    z = (1 - alpha) * z_a + alpha * z_b
    with torch.no_grad():
        logits = model.decode(z).cpu().numpy()[0]
    roll = logits_to_roll(logits)
    out_path = os.path.join(interp_dir, f"interp_{step+1}.mid")
    export_roll(roll, out_path)

def find_reference_midi():
    candidates = glob.glob(os.path.join("data", "raw_midi", "maestro-v3.0.0", "**", "*.mid"), recursive=True)
    return candidates[0] if candidates else None

def evaluate_folder(folder, ref_path):
    midi_files = sorted(glob.glob(os.path.join(folder, "*.mid")))
    rows = []
    for midi_path in midi_files:
        if ref_path:
            rows.append(evaluate_pair(ref_path, midi_path))
        else:
            rows.append({"pitch_hist": np.nan, "rhythm_diversity": np.nan, "repetition_ratio": np.nan})
    if not rows:
        return None
    return pd.DataFrame(rows).mean().to_dict()

ref_midi = find_reference_midi()
task1_dir = os.path.join("outputs", "generated_midis", "task1")
rows = []
for name, folder in [("Task1_LSTM", task1_dir), ("Task2_VAE", output_dir)]:
    if os.path.exists(folder):
        metrics = evaluate_folder(folder, ref_midi)
        if metrics:
            rows.append({"model": name, **metrics})
comparison_df = pd.DataFrame(rows)
comparison_df